# ADI/OCO sampling이 X21 / 회전 Y17에 미치는 영향
`C:\PHOTO_ANALYSIS\BASIC\CPE_Sampling_Diagnostics.ipynb`에 놓고 **00~13을 순서대로 실행**합니다. 별도 Python 모듈이나 DB 없이 저장된 stages를 읽습니다. 결과는 새 `output_cpe_sampling_diagnostics`에 저장합니다.

**비교 방향은 기존 실험과 같은 critical X입니다.** ADI와 OCO 각각 대표 wafer 하나를 선택하고 shot 내 좌표를 추출합니다. 측정 배치가 같은 mode 안에서는 대표 sampling으로 진단하며, ADI/OCO 간 다른 point를 1:1로 매칭하지 않습니다.

1. **02:** 대표 wafer/shot 좌표와 반복 layout 목록 저장. 기본은 가능한 같은 wafer에서 가장 완전한 반복 layout을 선택합니다. 00의 `WAFER_SELECTION`으로 원하는 wafer를 지정할 수 있습니다.
2. **03~07:** 원래 CPE 정의·solver·저장 ridge/G-opt/span 설정과 X21/회전Y17 대응표.
3. **08~09:** 공통 평가 grid, rank/항 식별성, alias와 노이즈 증폭 지도.
4. **10:** **동일한 정답 field**를 ADI와 OCO 측정 좌표에서 각각 관측한 것으로 가정하고, 공통 grid에서 복원 오차 비교. 각 단항식과 혼합 패턴을 시험합니다.
5. **11:** ADI는 Y17, OCO는 X21 우세가 같은 field에서도 재현되는지 확인. 입력에 X 값이 있으면 대표 shot의 실제 잔차와 공통항 대비 추가 설명력도 참고로 저장.
6. **12~13:** 그림과 `CPE_Sampling_Diagnostics.xlsx` 저장.

**좌표만으로 실제 오차 원인을 확정할 수는 없습니다.** 이 분석은 어떤 성분이 관측 가능하고, sampling과 제약 때문에 어떤 패턴에서 우열이 달라질 수 있는지 확인합니다. 합성 패턴의 빈도는 실제 원인의 확률이 아닙니다.

각 정답 패턴은 공통 grid의 RMS가 1 nm가 되도록 정규화합니다. noise 기본값 0.2 nm는 **가정값**이며 00에서 바꿀 수 있습니다. 공통 grid의 전체 영역과 두 sampling의 convex hull 교집합에서의 오차를 별도로 저장합니다. 학습점 잔차가 작아도 전체 field가 잘 복원된다는 뜻은 아니므로 `grid_rmse_nm`을 함께 보세요.


## 00. 경로 / 대표 wafer / 진단 설정


In [ ]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple
import hashlib
import json
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull, Delaunay, QhullError
from IPython.display import display, FileLink

BASE_DIR = Path(r"C:\PHOTO_ANALYSIS\BASIC")
STAGE_DIR = BASE_DIR / "output_decorrect_notebook" / "stages"
OUTPUT_DIR = BASE_DIR / "output_cpe_sampling_diagnostics"
MODES = ["ADI", "OCO"]
WAFER_SELECTION = {"ADI": None, "OCO": None}  # 예: {"LOT_ID": "LOT_A", "Wafer": 3}; None이면 자동 선정
FILTERS = {}  # 예: {"LOT_ID": ["LOT_A"]}; 입력 또는 wafer_context 컬럼으로 문제 제품 선택
USE_WAFER_CONTEXT = True
USE_VALID_X_MASK = True  # CPE 직전 X 컬럼이 있으면 값의 크기는 쓰지 않고 유효 point 조건만 맞춤
ROTATION_DEG = 90
USE_SAVED_SOLVER_SETTINGS = True
SOLVER_OVERRIDES = {}
COMMON_SHOT_SIZE_UM = None  # 예: (26000., 33000.); None이면 두 sampling의 pitch/좌표 범위 사용
COMMON_GRID_N = 25
COORD_ROUND_DECIMALS = 3  # sampling 패턴 식별에만 사용. 회귀에는 원래 좌표 사용
SVD_RCOND = 1e-10
ALIAS_COSINE_THRESHOLD = .98
SYNTHETIC_SIGNAL_RMS_NM = 1.0
ASSUMED_POINT_NOISE_NM = .2  # 실제 noise 추정값이 아닌 iid noise 가정. 0으로 끌 수 있음
RANDOM_MIXTURES = 12
RANDOM_SEED = 20261016
RUN_OBSERVED_CHECK = True
EXCEL_PREVIEW_ROWS = 10000
PREVIEW_ROWS = 8
SHOW_PLOTS = True
UM_TO_NM = 1000.
WAFER_KEYS = ["LOT_ID", "Wafer", "photo_transn_seq"]
SHOT_KEYS = WAFER_KEYS + ["fcp_x", "fcp_y"]
COORDS = ["coordinate_X", "coordinate_Y"]
GEOMETRY = ["STEP_PITCH_X", "STEP_PITCH_Y", "CHIP_X_NUM", "CHIP_Y_NUM"]
TARGET = "raw_minus_total_fit_x"
NATIVE_CASES = ["X21_NATIVE", "ROT_Y17_NATIVE"]
if ROTATION_DEG not in [90, -90] or COMMON_GRID_N < 5 or ASSUMED_POINT_NOISE_NM < 0 or SYNTHETIC_SIGNAL_RMS_NM <= 0:
    raise ValueError("회전각 / grid / signal / noise 설정을 확인하세요.")
if not STAGE_DIR.is_dir():
    raise FileNotFoundError(STAGE_DIR)
TABLE_DIR, PLOT_DIR = OUTPUT_DIR / "tables", OUTPUT_DIR / "plots"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
input_manifest, output_manifest = [], {}
plt.rcParams.update({"axes.unicode_minus": False, "figure.dpi": 110})
print("출력:", OUTPUT_DIR)
print("각 mode에서 대표 wafer 하나를 선택합니다. 비교 방향은 같은 critical X의 X21 / 회전 Y17입니다.")


## 01. 필요한 좌표 컬럼만 읽기 / 필터 / 중간 파일 저장


In [ ]:
def normalized_id(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    try:
        number = Decimal(text)
        return str(int(number)) if number.is_finite() and number == number.to_integral_value() else text
    except (InvalidOperation, ValueError, OverflowError):
        return text

def normalize_keys(frame):
    out = frame.copy()
    for key in WAFER_KEYS:
        values = out[key].astype("string").str.strip()
        if key != "LOT_ID":
            values = values.map({value: normalized_id(value) for value in values.dropna().unique()})
        out[key] = values.astype("string")
    return out

def excel_safe(frame):
    out = frame.copy()
    for col in out.select_dtypes(include=["object", "string"]).columns:
        out[col] = out[col].map(lambda value: "'"+value if isinstance(value, str) and value.startswith(("=", "+", "-", "@")) else value)
    return out

def save_table(name, frame, excel=True):
    frame.to_parquet(TABLE_DIR / f"{name}.parquet", index=False)
    frame.to_csv(TABLE_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
    if excel:
        suffix = f"_sample_{EXCEL_PREVIEW_ROWS}" if len(frame) > EXCEL_PREVIEW_ROWS else ""
        excel_safe(frame.head(EXCEL_PREVIEW_ROWS)).to_excel(TABLE_DIR / f"{name}{suffix}.xlsx", index=False)
    output_manifest[name] = dict(name=name, rows=len(frame), columns=len(frame.columns))
    print(f"[{name}] {len(frame):,} rows × {len(frame.columns)} columns")

def read_coordinates(mode):
    errors = []
    for stem in [f"21_{mode}_WK_RK_REMOVED", f"27_{mode}_FINAL_POINTS", f"15_{mode}_FINAL"]:
        for suffix in [".parquet", ".xlsx"]:
            path = STAGE_DIR / (stem+suffix)
            if not path.is_file():
                continue
            try:
                if suffix == ".parquet":
                    names = pq.read_schema(path).names
                    columns = list(dict.fromkeys(WAFER_KEYS+SHOT_KEYS[3:]+COORDS+GEOMETRY+[TARGET]+list(FILTERS)))
                    data = pd.read_parquet(path, columns=[col for col in columns if col in names])
                else:
                    data = pd.read_excel(path, dtype={key: "string" for key in WAFER_KEYS})
                missing = [col for col in SHOT_KEYS+COORDS if col not in data]
                if missing:
                    raise ValueError(f"좌표/wafer 키 컬럼 누락: {missing}")
                data = normalize_keys(data)
                for col in SHOT_KEYS[3:]+COORDS+GEOMETRY+[TARGET]:
                    if col in data:
                        data[col] = pd.to_numeric(data[col], errors="coerce")
                valid = data[WAFER_KEYS].notna().all(axis=1) & np.isfinite(data[SHOT_KEYS[3:]+COORDS].to_numpy(dtype=float,na_value=np.nan)).all(axis=1)
                if USE_VALID_X_MASK and TARGET in data:
                    valid &= np.isfinite(data[TARGET])
                input_manifest.append(dict(mode=mode, path=str(path), rows=len(data), valid_rows=int(valid.sum()), x_validity_mask_used=USE_VALID_X_MASK and TARGET in data, bytes=path.stat().st_size, mtime_ns=path.stat().st_mtime_ns, fallback_errors="; ".join(errors)))
                return data.loc[valid].reset_index(drop=True)
            except Exception as exc:
                errors.append(f"{path.name}: {type(exc).__name__}: {exc}")
    raise FileNotFoundError(f"{mode}: 전체 Parquet/Excel 좌표 입력 필요. sample 파일은 제외. {'; '.join(errors)}")

def filter_frame(frame, rules):
    out = frame
    for col, values in rules.items():
        if col not in out:
            raise ValueError(f"필터 컬럼 없음: {col}")
        if not isinstance(values, (list, tuple, set)):
            values = [values]
        if not values:
            raise ValueError(f"비어 있는 필터: {col}")
        values = [str(normalized_id(v)) if col in WAFER_KEYS[1:] else str(v).strip() for v in values]
        out = out.loc[out[col].astype("string").str.strip().isin(values)]
    return out


## 02. 대표 wafer와 반복 sampling 패턴 추출


In [ ]:
frames = {mode: read_coordinates(mode) for mode in MODES}
if USE_WAFER_CONTEXT and FILTERS:
    context_path = next((BASE_DIR / "output_wafer_context" / ("wafer_context"+suffix) for suffix in [".parquet", ".xlsx"] if (BASE_DIR / "output_wafer_context" / ("wafer_context"+suffix)).is_file()), None)
    if context_path:
        context = normalize_keys(pd.read_parquet(context_path) if context_path.suffix == ".parquet" else pd.read_excel(context_path, dtype={key:"string" for key in WAFER_KEYS}))
        if context.duplicated(WAFER_KEYS).any() or context[WAFER_KEYS].isna().any(axis=None):
            raise ValueError("wafer_context 키가 유일하고 유효해야 합니다.")
        for mode in MODES:
            extra = [col for col in FILTERS if col not in frames[mode] and col in context]
            if extra:
                frames[mode] = frames[mode].merge(context[WAFER_KEYS+extra], on=WAFER_KEYS, how="left", validate="many_to_one")
for mode in MODES:
    frames[mode] = filter_frame(filter_frame(frames[mode], FILTERS), WAFER_SELECTION[mode] or {})
    if frames[mode].empty:
        raise ValueError(f"{mode}: 선택 가능한 유효 좌표가 없습니다.")
catalogues = {mode: frame.groupby(WAFER_KEYS, sort=False).size().rename("n_valid_rows").reset_index() for mode,frame in frames.items()}
common = catalogues["ADI"].merge(catalogues["OCO"], on=WAFER_KEYS, suffixes=("_ADI", "_OCO"))
if len(common):
    common["paired_size"] = common[["n_valid_rows_ADI", "n_valid_rows_OCO"]].min(axis=1)
    common_key = common.sort_values("paired_size", ascending=False, kind="stable").iloc[0][WAFER_KEYS].tolist()
else:
    common_key = None
selected_rows, layout_rows, shot_rows, primary_frames, wafer_frames = [], [], [], {}, {}
for mode in MODES:
    key = common_key or catalogues[mode].sort_values("n_valid_rows", ascending=False, kind="stable").iloc[0][WAFER_KEYS].tolist()
    frame = frames[mode].loc[frames[mode][WAFER_KEYS].eq(pd.Series(key, index=WAFER_KEYS)).all(axis=1)].copy()
    wafer_frames[mode] = frame
    selected_rows.append(dict(mode=mode, **dict(zip(WAFER_KEYS,key)), n_points=len(frame), common_wafer=common_key is not None))
    groups, signatures = {}, {}
    for shot_key, work in frame.groupby(["fcp_x", "fcp_y"], sort=False):
        rounded = work[COORDS].round(COORD_ROUND_DECIMALS).sort_values(COORDS).to_numpy(dtype="<f8")
        signature = hashlib.sha256(rounded.tobytes()).hexdigest()[:16]
        groups[shot_key] = work
        signatures.setdefault(signature, []).append(shot_key)
    choices = []
    for signature, keys in signatures.items():
        representative = min(keys, key=lambda key: key[0]**2+key[1]**2)
        work = groups[representative]
        row = dict(mode=mode, layout_id=signature, n_shots=len(keys), n_points=len(work), n_unique_points=len(work[COORDS].drop_duplicates()), fcp_x=representative[0], fcp_y=representative[1])
        choices.append(row)
    best = max(choices, key=lambda row: (row["n_unique_points"], row["n_shots"], -row["fcp_x"]**2-row["fcp_y"]**2))
    primary_frames[mode] = groups[(best["fcp_x"], best["fcp_y"])].copy()
    for row in choices:
        row["is_primary"] = row["layout_id"] == best["layout_id"]
        layout_rows.append(row)
    for signature, keys in signatures.items():
        shot_rows.extend(dict(mode=mode, fcp_x=x, fcp_y=y, layout_id=signature, is_primary=signature == best["layout_id"]) for x,y in keys)
    print(mode, "대표 wafer:", key, "; 대표 shot:", (best["fcp_x"], best["fcp_y"]), "; mark 수:", best["n_points"])
selection, layout_catalogue, shot_layouts = pd.DataFrame(selected_rows), pd.DataFrame(layout_rows), pd.DataFrame(shot_rows)
selected_points = pd.concat([frame.assign(mode=mode) for mode,frame in wafer_frames.items()], ignore_index=True)
for name, table in [("02_SELECTED_WAFERS", selection), ("02_LAYOUT_CATALOGUE", layout_catalogue), ("02_SHOT_LAYOUTS", shot_layouts), ("02_SELECTED_COORDINATES", selected_points)]:
    save_table(name, table)
del frames
display(selection)


## 03. CPE38 설정, 행렬 및 span 제한
원본 solver 정의입니다. 07에서 저장 설정과 override를 적용합니다.


In [ ]:
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

SPAN_MODE = "OVO3_38P_F2F"
RIDGE_REGRESSION = True
G_OPT_SPEC = 1.0
USE_VIRTUAL_CHIP_POINTS = True   # VBA virtual grid: chip별 4 corner + center point
G_OPT_TOLERANCE = 0.001          # VBA: G_opt_tolerlance = 0.001
SPAN_TOLERANCE = 0.01            # VBA: span_tolerlance = 0.01
MINIMUM_RIDGE = 0.001            # VBA: minimum_Ridge = 0.001
ADD_RIDGE = 10.0                 # VBA: add_ridge = 10
MAX_GOPT_RETURN = 50             # VBA: max_prevent_fallback_retrun = 50
MAX_SPAN_ITER = 5000             # VBA: max_iter = 5000
CD_TOL = 1e-6                    # VBA: tol = 0.000001
CPE_EVAL_GRID_N = 10
MIN_POINTS_GUIDE = 10
G_OPTIMAL_TYPE = 1               # VBA information 기준: GOT1
VIRTUAL_WAFER_RADIUS = 144000.0  # VBA virtual point keep 조건: R <= 144000               # VBA option D59: 1,2,3,4. 4 = inner OT2 / partial OT1
USE_SPAN_BOX_CONSTRAINT = True   # VBA Balloon projected CD/PGD stage

def cpe_38para(rx: Sequence[float], ry: Sequence[float]) -> Tuple[np.ndarray, np.ndarray]:
    rx = np.asarray(rx, dtype=float)
    ry = np.asarray(ry, dtype=float)
    X_dx = np.vstack([
        np.ones(len(rx)),
        rx / 1e6, -ry / 1e6,
        (rx ** 2) / 1e9, (rx * ry) / 1e9, (ry ** 2) / 1e9,
        (rx ** 3) / 1e12, (rx ** 2 * ry) / 1e12, (rx * ry ** 2) / 1e12, (ry ** 3) / 1e12,
        (rx ** 3 * ry) / 1e19, (rx ** 2 * ry ** 2) / 1e19, (rx * ry ** 3) / 1e19, (ry ** 4) / 1e19,
        (rx ** 3 * ry ** 2) / 1e23, (rx ** 2 * ry ** 3) / 1e23, (rx * ry ** 4) / 1e23, (ry ** 5) / 1e23,
        (rx ** 3 * ry ** 3) / 1e27, (rx ** 2 * ry ** 4) / 1e27,
        (rx ** 3 * ry ** 4) / 1e31,
    ]).T
    X_dy = np.vstack([
        np.ones(len(ry)),
        ry / 1e6, rx / 1e6,
        (ry ** 2) / 1e9, (ry * rx) / 1e9, (rx ** 2) / 1e9,
        (ry ** 3) / 1e12, (ry ** 2 * rx) / 1e12, (ry * rx ** 2) / 1e12,
        (ry ** 4) / 1e19, (ry ** 3 * rx) / 1e19, (ry ** 2 * rx ** 2) / 1e19,
        (ry ** 5) / 1e23, (ry ** 4 * rx) / 1e23, (ry ** 3 * rx ** 2) / 1e23,
        (ry ** 5 * rx) / 1e27, (ry ** 4 * rx ** 2) / 1e27,
    ]).T
    return X_dx, X_dy
CPE38_KEYS_DX = [
    "RK1", "RK3", "RK5",
    "RK7", "RK9", "RK11",
    "RK13", "RK15", "RK17", "RK19",
    "RK23", "RK25", "RK27", "RK29",
    "RK35", "RK37", "RK39", "RK41",
    "RK49", "RK51",
    "RK65",
]
CPE38_KEYS_DY = [
    "RK2", "RK4", "RK6",
    "RK8", "RK10", "RK12",
    "RK14", "RK16", "RK18",
    "RK22", "RK24", "RK26",
    "RK32", "RK34", "RK36",
    "RK46", "RK48",
]
RK_SPAN_LIMITS_OVO3_38P_F2F = {
    'RK1': {'min': -0.1, 'max': 0.1},
    'RK2': {'min': -0.1, 'max': 0.1},
    'RK3': {'min': -2.0, 'max': 2.0},
    'RK4': {'min': -10.0, 'max': 10.0},
    'RK5': {'min': -5.0, 'max': 5.0},
    'RK6': {'min': -5.0, 'max': 5.0},
    'RK7': {'min': -0.1, 'max': 0.1},
    'RK8': {'min': -0.15, 'max': 0.15},
    'RK9': {'min': -0.06, 'max': 0.06},
    'RK10': {'min': -0.5, 'max': 0.5},
    'RK11': {'min': -0.5, 'max': 0.5},
    'RK12': {'min': -0.03, 'max': 0.03},
    'RK13': {'min': -0.003, 'max': 0.003},
    'RK14': {'min': -0.015, 'max': 0.015},
    'RK15': {'min': -0.002, 'max': 0.002},
    'RK16': {'min': -0.05, 'max': 0.05},
    'RK17': {'min': -0.005, 'max': 0.005},
    'RK18': {'min': -0.002, 'max': 0.002},
    'RK19': {'min': -0.023, 'max': 0.023},
    'RK22': {'min': -7.0, 'max': 7.0},
    'RK23': {'min': -2.5, 'max': 2.5},
    'RK24': {'min': -18.0, 'max': 18.0},
    'RK25': {'min': -8.0, 'max': 8.0},
    'RK26': {'min': -4.0, 'max': 4.0},
    'RK27': {'min': -7.0, 'max': 7.0},
    'RK29': {'min': -7.0, 'max': 7.0},
    'RK32': {'min': -2.5, 'max': 2.5},
    'RK34': {'min': -5.0, 'max': 5.0},
    'RK35': {'min': -2.0, 'max': 2.0},
    'RK36': {'min': -1.2, 'max': 1.2},
    'RK37': {'min': -2.5, 'max': 2.5},
    'RK39': {'min': -4.0, 'max': 4.0},
    'RK41': {'min': -2.5, 'max': 2.5},
    'RK46': {'min': -1.2, 'max': 1.2},
    'RK48': {'min': -0.6, 'max': 0.6},
    'RK49': {'min': -0.8, 'max': 0.8},
    'RK51': {'min': -1.2, 'max': 1.2},
    'RK65': {'min': -0.7, 'max': 0.7},
}
RK_SPAN_LIMITS_OVO3_38P_HARDLIMIT = {
    'RK1': {'min': -100.0, 'max': 100.0},
    'RK2': {'min': -100.0, 'max': 100.0},
    'RK3': {'min': -1000.0, 'max': 1000.0},
    'RK4': {'min': -1000.0, 'max': 1000.0},
    'RK5': {'min': -1000.0, 'max': 1000.0},
    'RK6': {'min': -1000.0, 'max': 1000.0},
    'RK7': {'min': -0.25, 'max': 0.25},
    'RK8': {'min': -10.0, 'max': 10.0},
    'RK9': {'min': -0.06, 'max': 0.06},
    'RK10': {'min': -10.0, 'max': 10.0},
    'RK11': {'min': -10.0, 'max': 10.0},
    'RK12': {'min': -0.25, 'max': 0.25},
    'RK13': {'min': -0.025, 'max': 0.025},
    'RK14': {'min': -1.0, 'max': 1.0},
    'RK15': {'min': -0.002, 'max': 0.002},
    'RK16': {'min': -1.0, 'max': 1.0},
    'RK17': {'min': -0.005, 'max': 0.005},
    'RK18': {'min': -0.002, 'max': 0.002},
    'RK19': {'min': -1.0, 'max': 1.0},
    'RK22': {'min': -7.0, 'max': 7.0},
    'RK23': {'min': -2.5, 'max': 2.5},
    'RK24': {'min': -18.0, 'max': 18.0},
    'RK25': {'min': -8.0, 'max': 8.0},
    'RK26': {'min': -4.0, 'max': 4.0},
    'RK27': {'min': -7.0, 'max': 7.0},
    'RK29': {'min': -7.0, 'max': 7.0},
    'RK32': {'min': -2.5, 'max': 2.5},
    'RK34': {'min': -5.0, 'max': 5.0},
    'RK35': {'min': -2.0, 'max': 2.0},
    'RK36': {'min': -1.2, 'max': 1.2},
    'RK37': {'min': -2.5, 'max': 2.5},
    'RK39': {'min': -4.0, 'max': 4.0},
    'RK41': {'min': -2.5, 'max': 2.5},
    'RK46': {'min': -1.2, 'max': 1.2},
    'RK48': {'min': -0.6, 'max': 0.6},
    'RK49': {'min': -0.8, 'max': 0.8},
    'RK51': {'min': -1.2, 'max': 1.2},
    'RK65': {'min': -0.7, 'max': 0.7},
}

def get_active_span_limits(span_mode: str = "OVO3_38P_F2F") -> Dict[str, Dict[str, float]]:
    if span_mode == "OVO3_38P_HARDLIMIT":
        return RK_SPAN_LIMITS_OVO3_38P_HARDLIMIT
    return RK_SPAN_LIMITS_OVO3_38P_F2F
RIDGE_VALVE_HYPER = {
    "RK1": 0.0, "RK2": 0.0,
    "RK3": 0.01, "RK4": 0.01, "RK5": 0.01, "RK6": 0.01,
    "RK7": 1.0, "RK8": 0.1, "RK9": 1.0, "RK10": 0.1,
    "RK11": 0.1, "RK12": 1.0,
    "RK13": 1.0, "RK14": 0.1, "RK15": 1.0, "RK16": 0.1,
    "RK17": 1.0, "RK18": 1.0, "RK19": 0.1, "RK20": 1.0,
}


## 04. CPE virtual grid 및 소표본 ridge 배율
원본 solver 정의입니다. 07에서 저장 설정과 override를 적용합니다.


In [ ]:
def _to_numeric(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def choose_base_lambda_scale(n_points: int) -> float:
    """VBA의 point가 적어도 skip하지 않는 방식을 Python식으로 반영."""
    if n_points >= 10:
        return 1.0
    if n_points >= 7:
        return 1e2
    if n_points >= 5:
        return 1e4
    if n_points >= 3:
        return 1e6
    return 1e8

def get_ridge_valves(coeff_names: Sequence[str]) -> np.ndarray:
    return np.array([RIDGE_VALVE_HYPER.get(name, 1.0) for name in coeff_names], dtype=float)

def make_lambda_vector(base_lambda: float, coeff_names: Sequence[str]) -> np.ndarray:
    valves = get_ridge_valves(coeff_names)
    return float(base_lambda) * valves

def build_virtual_chip_points(
    step_pitch_x: float,
    step_pitch_y: float,
    chip_x_num: int,
    chip_y_num: int,
    round_decimals: int = 6,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    VBA virtual grid 재현:
    chip별 4 corner + center point를 shot-local coordinate로 생성하고,
    chip 경계에서 겹치는 corner는 dedup한다.
    예: chip_x_num=2, chip_y_num=1
      - 각 chip 5 point => 10 raw points
      - 가운데 공유 corner 2개 중복 제거
      - 최종 8 virtual points
    coordinate_X/Y가 shot center 기준이라고 보고,
    전체 shot 영역은 [-STEP_PITCH_X/2, +STEP_PITCH_X/2],
                  [-STEP_PITCH_Y/2, +STEP_PITCH_Y/2] 로 둔다.
    """
    chip_x_num = int(chip_x_num) if pd.notna(chip_x_num) and int(chip_x_num) > 0 else 1
    chip_y_num = int(chip_y_num) if pd.notna(chip_y_num) and int(chip_y_num) > 0 else 1
    sx = float(step_pitch_x)
    sy = float(step_pitch_y)
    x_edges = np.linspace(-sx / 2.0, sx / 2.0, chip_x_num + 1)
    y_edges = np.linspace(-sy / 2.0, sy / 2.0, chip_y_num + 1)
    pts = []
    for ix in range(chip_x_num):
        for iy in range(chip_y_num):
            x0, x1 = x_edges[ix], x_edges[ix + 1]
            y0, y1 = y_edges[iy], y_edges[iy + 1]
            xc = (x0 + x1) / 2.0
            yc = (y0 + y1) / 2.0
            pts.extend([
                (x0, y0), (x0, y1), (x1, y0), (x1, y1),
                (xc, yc),
            ])
    pts_unique = sorted({(round(x, round_decimals), round(y, round_decimals)) for x, y in pts})
    rx = np.array([p[0] for p in pts_unique], dtype=float)
    ry = np.array([p[1] for p in pts_unique], dtype=float)
    return rx, ry

def filter_virtual_points_by_wafer_radius(
    rx_v: Sequence[float],
    ry_v: Sequence[float],
    fcp_x: float,
    fcp_y: float,
    wafer_radius: float = VIRTUAL_WAFER_RADIUS,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    VBA edge-shot virtual point filtering.
    virtual point의 wafer 좌표 R이 wafer_radius 이하인 point만 사용한다.
    R = sqrt((fcp_x + rx_v)^2 + (fcp_y + ry_v)^2)
    keep if R <= 144000
    """
    rx_v = np.asarray(rx_v, dtype=float)
    ry_v = np.asarray(ry_v, dtype=float)
    wx = float(fcp_x) + rx_v
    wy = float(fcp_y) + ry_v
    r = np.sqrt(wx * wx + wy * wy)
    keep = r <= float(wafer_radius)
    return rx_v[keep], ry_v[keep], r[keep]

def get_virtual_points_for_group(
    grp: pd.DataFrame,
    fallback_rx: Sequence[float],
    fallback_ry: Sequence[float],
    fcp_x: float,
    fcp_y: float,
    wafer_radius: float = VIRTUAL_WAFER_RADIUS,
) -> Tuple[np.ndarray, np.ndarray, str, int, int]:
    """
    1) full virtual point 생성: chip별 4 corner + center
    2) edge shot 처리: wafer 좌표 R <= wafer_radius point만 keep
    반환:
      rx_v, ry_v, virtual_source, n_virtual_full, n_virtual_kept
    """
    required = ["STEP_PITCH_X", "STEP_PITCH_Y", "CHIP_X_NUM", "CHIP_Y_NUM"]
    if all(c in grp.columns for c in required):
        vals = grp[required].dropna()
        if len(vals) > 0:
            row = vals.iloc[0]
            rx_full, ry_full = build_virtual_chip_points(
                step_pitch_x=row["STEP_PITCH_X"],
                step_pitch_y=row["STEP_PITCH_Y"],
                chip_x_num=row["CHIP_X_NUM"],
                chip_y_num=row["CHIP_Y_NUM"],
            )
            n_full = len(rx_full)
            rx_keep, ry_keep, _ = filter_virtual_points_by_wafer_radius(
                rx_full,
                ry_full,
                fcp_x=fcp_x,
                fcp_y=fcp_y,
                wafer_radius=wafer_radius,
            )
            if len(rx_keep) > 0:
                src = "chip_corner_center_wafer_clipped"
                return rx_keep, ry_keep, src, n_full, len(rx_keep)
            return rx_full, ry_full, "chip_corner_center_no_clip_points_fallback", n_full, n_full
    rx_v, ry_v = build_eval_grid_from_data(fallback_rx, fallback_ry, grid_n=CPE_EVAL_GRID_N)
    return rx_v, ry_v, "measured_range_grid_fallback", len(rx_v), len(rx_v)


## 05. Ridge 예측 및 G-opt 평가 함수
원본 solver 정의입니다. 07에서 저장 설정과 override를 적용합니다.


In [ ]:
def stable_inverse_or_pinv(A: np.ndarray) -> np.ndarray:
    try:
        return np.linalg.inv(A)
    except np.linalg.LinAlgError:
        return np.linalg.pinv(A)

def ridge_predict(X: np.ndarray, y: np.ndarray, lambdas: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    A = X.T @ X + np.diag(lambdas)
    b = X.T @ y
    try:
        beta = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        beta = np.linalg.pinv(A) @ b
    return beta, X @ beta

def build_eval_grid_from_data(rx: Sequence[float], ry: Sequence[float], grid_n: int = 10) -> Tuple[np.ndarray, np.ndarray]:
    rx = np.asarray(rx, dtype=float)
    ry = np.asarray(ry, dtype=float)
    rx_min, rx_max = np.nanmin(rx), np.nanmax(rx)
    ry_min, ry_max = np.nanmin(ry), np.nanmax(ry)
    if not np.isfinite(rx_min) or not np.isfinite(rx_max):
        rx_min, rx_max = -1.0, 1.0
    if not np.isfinite(ry_min) or not np.isfinite(ry_max):
        ry_min, ry_max = -1.0, 1.0
    if rx_min == rx_max:
        rx_min -= 1.0
        rx_max += 1.0
    if ry_min == ry_max:
        ry_min -= 1.0
        ry_max += 1.0
    gx = np.linspace(rx_min, rx_max, grid_n)
    gy = np.linspace(ry_min, ry_max, grid_n)
    RX, RY = np.meshgrid(gx, gy)
    return RX.ravel(), RY.ravel()

def gopt_value(
    X_meas: np.ndarray,
    X_eval: np.ndarray,
    lambdas: np.ndarray,
    gopt_type: int = 4,
    inner_or_partial: str = "partial",
) -> float:
    """
    VBA G_optimal_type 반영.
    - Type1: H H.T diag의 sqrt(max)
    - Type2: H matrix element의 signed max의 sqrt. 음수이면 999.0 반환.
    - Type3: X_eval inv(...) X_eval.T diag의 sqrt(max)
    - Type4: inner=Type2, partial=Type1
    """
    X_meas = np.asarray(X_meas, dtype=float)
    X_eval = np.asarray(X_eval, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    A = X_meas.T @ X_meas + np.diag(lambdas)
    invA = stable_inverse_or_pinv(A)
    effective_type = gopt_type
    if gopt_type == 4:
        effective_type = 2 if inner_or_partial == "inner" else 1
    if effective_type == 1:
        H = X_eval @ invA @ X_meas.T
        diag_val = np.sum(H * H, axis=1)
        max_val = np.nanmax(diag_val)
        return float(np.sqrt(max(max_val, 0.0)))
    if effective_type == 2:
        H = X_eval @ invA @ X_meas.T
        max_val = float(np.nanmax(H))
        if max_val < 0:
            return 999.0
        return float(np.sqrt(max_val))
    if effective_type == 3:
        M = X_eval @ invA @ X_eval.T
        diag_val = np.diag(M)
        max_val = np.nanmax(diag_val)
        return float(np.sqrt(max(max_val, 0.0)))
    raise ValueError(f"Unsupported gopt_type: {gopt_type}")


## 06. Span/box 제약 및 한 방향 회귀 solver
원본 solver 정의입니다. 07에서 저장 설정과 override를 적용합니다.


In [ ]:
def calc_contribution_spans(X_eval: np.ndarray, beta: np.ndarray, coeff_names: Sequence[str]) -> Dict[str, float]:
    spans = {}
    for i, name in enumerate(coeff_names):
        contrib = X_eval[:, i] * beta[i]
        spans[name] = float(np.nanmax(contrib) - np.nanmin(contrib))
    return spans

def find_span_violations(
    spans: Dict[str, float],
    span_limits: Dict[str, Dict[str, float]],
    tolerance: float = 0.01,
) -> List[Tuple[str, float, float, float]]:
    violations = []
    for name, span in spans.items():
        if name not in span_limits:
            continue
        lo = span_limits[name]["min"] - tolerance
        hi = span_limits[name]["max"] + tolerance
        if span < lo or span > hi:
            violations.append((name, span, span_limits[name]["min"], span_limits[name]["max"]))
    return violations

def build_beta_bounds_from_span(
    X_eval: np.ndarray,
    coeff_names: Sequence[str],
    span_limits: Dict[str, Dict[str, float]],
) -> Tuple[np.ndarray, np.ndarray]:
    """
    VBA의 min_Span_X/Y, max_Span_X/Y에 대응하는 box constraint 근사.
    contribution span = range(X_j * beta_j)가 span_limit 안에 들어오도록
    beta_j bound를 계산한다.
    """
    n = len(coeff_names)
    lo = np.full(n, -np.inf, dtype=float)
    hi = np.full(n, np.inf, dtype=float)
    for j, name in enumerate(coeff_names):
        if name not in span_limits:
            continue
        lim = span_limits[name]
        span_abs = max(abs(float(lim["min"])), abs(float(lim["max"])))
        xj_range = float(np.nanmax(X_eval[:, j]) - np.nanmin(X_eval[:, j]))
        if xj_range <= 0 or not np.isfinite(xj_range):
            continue
        b = span_abs / xj_range
        lo[j] = -b
        hi[j] = b
    return lo, hi

def projected_coordinate_descent_ridge(
    X: np.ndarray,
    y: np.ndarray,
    lambdas: np.ndarray,
    beta_init: Optional[np.ndarray] = None,
    lower: Optional[np.ndarray] = None,
    upper: Optional[np.ndarray] = None,
    tol: float = 1e-6,
    max_iter: int = 5000,
) -> Tuple[np.ndarray, np.ndarray, int, float]:
    """
    VBA ProjectedCoordinateDescentElasticNet(l1=0) 근사.
    solve min ||y-Xb||^2 + sum(lambda_j*b_j^2), s.t. lower <= b <= upper
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    n_param = X.shape[1]
    if beta_init is None:
        beta, _ = ridge_predict(X, y, lambdas)
    else:
        beta = np.asarray(beta_init, dtype=float).copy()
    if lower is None:
        lower = np.full(n_param, -np.inf)
    if upper is None:
        upper = np.full(n_param, np.inf)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    beta = np.clip(beta, lower, upper)
    residual = y - X @ beta
    col_norm2 = np.sum(X * X, axis=0)
    last_delta = np.inf
    for it in range(1, max_iter + 1):
        max_delta = 0.0
        for j in range(n_param):
            denom = col_norm2[j] + lambdas[j]
            if denom <= 0 or not np.isfinite(denom):
                continue
            residual += X[:, j] * beta[j]
            rho = float(X[:, j] @ residual)
            new_beta = rho / denom
            new_beta = min(max(new_beta, lower[j]), upper[j])
            delta = abs(new_beta - beta[j])
            beta[j] = new_beta
            residual -= X[:, j] * beta[j]
            if delta > max_delta:
                max_delta = delta
        last_delta = max_delta
        if max_delta < tol:
            break
    return beta, X @ beta, it, float(last_delta)

@dataclass
class RidgeFitResult:
    beta: np.ndarray
    pred: np.ndarray
    lambdas: np.ndarray
    gopt: float
    spans: Dict[str, float]
    violations: List[Tuple[str, float, float, float]]
    gopt_iter: int
    span_iter: int
    method: str

def solve_vba_style_ridge_one_axis(
    X_fit: np.ndarray,
    y: np.ndarray,
    X_eval: np.ndarray,
    coeff_names: Sequence[str],
    span_limits: Dict[str, Dict[str, float]],
    ridge_regression: bool = True,
    gopt_spec: float = 1.0,
    gopt_tolerance: float = 0.001,
    minimum_ridge: float = 0.001,
    add_ridge: float = 10.0,
    max_gopt_return: int = 50,
    gopt_type: int = 4,
    inner_or_partial: str = "partial",
    use_span_box_constraint: bool = True,
    span_tolerance: float = 0.01,
    cd_tol: float = 1e-6,
    max_span_iter: int = 5000,
    base_lambda_scale: float = 1.0,
    debug: bool = False,
) -> RidgeFitResult:
    """
    VBA flow에 맞춘 한 방향(X or Y) fitting.
    - 초기 ridge는 0에 가까운 값이 아니라 VBA minimum_Ridge 기준 사용
    - Gopt가 spec 이상이면 ridge를 빠르게 키운 뒤 이분탐색으로 spec 근처를 찾음
    - span over가 남으면 projected coordinate descent로 coefficient를 box constraint 안에 넣음
    """
    X_fit = np.asarray(X_fit, dtype=float)
    X_eval = np.asarray(X_eval, dtype=float)
    y = np.asarray(y, dtype=float)
    n_param = X_fit.shape[1]
    if n_param != len(coeff_names):
        raise ValueError(f"n_param({n_param}) != len(coeff_names)({len(coeff_names)})")
    initial_lambda = 0.0 if not ridge_regression else 0.0
    lambdas = make_lambda_vector(initial_lambda, coeff_names)
    beta, pred = ridge_predict(X_fit, y, lambdas)
    g = gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
    gopt_iter = 0
    method = "ols"
    if ridge_regression and (not np.isfinite(g) or g >= gopt_spec or g == 0):
        low_lambda = 0.0
        high_lambda = max(minimum_ridge * base_lambda_scale, minimum_ridge)
        high_lambda = max(high_lambda, minimum_ridge)
        high_lambdas = make_lambda_vector(high_lambda, coeff_names)
        high_g = gopt_value(X_fit, X_eval, high_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        while (not np.isfinite(high_g) or high_g >= gopt_spec) and gopt_iter < max_gopt_return:
            low_lambda = high_lambda
            high_lambda = high_lambda * add_ridge
            high_lambdas = make_lambda_vector(high_lambda, coeff_names)
            high_g = gopt_value(X_fit, X_eval, high_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
            gopt_iter += 1
            if debug:
                print(f"fast ridge: lambda={high_lambda:.6g}, gopt={high_g:.6g}")
        target_low = max(0.0, gopt_spec - gopt_tolerance)
        target_high = gopt_spec
        best_lambda = high_lambda
        best_g = high_g
        if np.isfinite(high_g) and high_g < gopt_spec:
            lo = low_lambda
            hi = high_lambda
            for _ in range(max_gopt_return):
                mid = (lo + hi) / 2.0
                mid_lambdas = make_lambda_vector(mid, coeff_names)
                mid_g = gopt_value(X_fit, X_eval, mid_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
                gopt_iter += 1
                if debug:
                    print(f"slow ridge: lambda={mid:.6g}, gopt={mid_g:.6g}")
                if target_low <= mid_g < target_high:
                    best_lambda = mid
                    best_g = mid_g
                    break
                if mid_g >= gopt_spec or not np.isfinite(mid_g):
                    lo = mid
                else:
                    hi = mid
                    best_lambda = mid
                    best_g = mid_g
                if abs(hi - lo) < max(minimum_ridge * 1e-3, 1e-12):
                    break
        lambdas = make_lambda_vector(best_lambda, coeff_names)
        beta, pred = ridge_predict(X_fit, y, lambdas)
        g = gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        method = "ridge_gopt"
    spans = calc_contribution_spans(X_eval, beta, coeff_names)
    violations = find_span_violations(spans, span_limits, tolerance=span_tolerance)
    span_iter = 0
    if ridge_regression and use_span_box_constraint and len(violations) > 0:
        lower, upper = build_beta_bounds_from_span(X_eval, coeff_names, span_limits)
        beta_cd, pred_cd, span_iter, _ = projected_coordinate_descent_ridge(
            X_fit,
            y,
            lambdas,
            beta_init=beta,
            lower=lower,
            upper=upper,
            tol=cd_tol,
            max_iter=max_span_iter,
        )
        beta = beta_cd
        pred = pred_cd
        spans = calc_contribution_spans(X_eval, beta, coeff_names)
        violations = find_span_violations(spans, span_limits, tolerance=span_tolerance)
        g = gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        method = "ridge_gopt_box_cd"
    return RidgeFitResult(
        beta=beta,
        pred=pred,
        lambdas=lambdas,
        gopt=float(g),
        spans=spans,
        violations=violations,
        gopt_iter=int(gopt_iter),
        span_iter=int(span_iter),
        method=method,
    )

def determine_inner_or_partial(n_valid: int, n_virtual: int) -> str:
    """VBA: virtual_points = Vgrid_XY면 inner, 아니면 partial."""
    return "inner" if n_valid >= n_virtual else "partial"


## 07. 회전 설정 / 모델 대응표 / X 한 방향 solver


In [ ]:
# 03~06의 원본 solver 정의를 먼저 실행합니다.
SETTINGS_MAP = {
    "span_mode": "SPAN_MODE", "min_points_guide": "MIN_POINTS_GUIDE", "eval_grid_n": "CPE_EVAL_GRID_N", "ridge_regression": "RIDGE_REGRESSION",
    "gopt_spec": "G_OPT_SPEC", "gopt_tolerance": "G_OPT_TOLERANCE", "gopt_type": "G_OPTIMAL_TYPE", "use_span_box_constraint": "USE_SPAN_BOX_CONSTRAINT",
    "use_virtual_chip_points": "USE_VIRTUAL_CHIP_POINTS", "virtual_wafer_radius_um": "VIRTUAL_WAFER_RADIUS", "span_tolerance": "SPAN_TOLERANCE",
    "minimum_ridge": "MINIMUM_RIDGE", "add_ridge": "ADD_RIDGE", "max_gopt_return": "MAX_GOPT_RETURN", "max_span_iter": "MAX_SPAN_ITER", "cd_tol": "CD_TOL", "ridge_valves": "RIDGE_VALVE_HYPER",
}
settings_path = STAGE_DIR.parent / "extension_settings.json"
saved_settings = json.loads(settings_path.read_text(encoding="utf-8")) if USE_SAVED_SOLVER_SETTINGS and settings_path.is_file() else {}
for source, value in {**saved_settings, **saved_settings.get("cpe_options", {})}.items():
    if source in SETTINGS_MAP:
        globals()[SETTINGS_MAP[source]] = value
NATIVE_SPAN_LIMITS = saved_settings.get("active_span_limits", get_active_span_limits(SPAN_MODE))
for key, value in SOLVER_OVERRIDES.items():
    if key not in set(SETTINGS_MAP.values()) | {"NATIVE_SPAN_LIMITS"}:
        raise ValueError(f"SOLVER_OVERRIDES: 지원하지 않는 설정 {key}")
    globals()[key] = value
if "SPAN_MODE" in SOLVER_OVERRIDES and "NATIVE_SPAN_LIMITS" not in SOLVER_OVERRIDES:
    NATIVE_SPAN_LIMITS = get_active_span_limits(SPAN_MODE)
ROTATION_SIGN = 1 if ROTATION_DEG == 90 else -1
X_POWERS = [(0,0), (1,0), (0,1), (2,0), (1,1), (0,2), (3,0), (2,1), (1,2), (0,3), (3,1), (2,2), (1,3), (0,4), (3,2), (2,3), (1,4), (0,5), (3,3), (2,4), (3,4)]
Y_POWERS = [(0,0), (0,1), (1,0), (0,2), (1,1), (2,0), (0,3), (1,2), (2,1), (0,4), (1,3), (2,2), (0,5), (1,4), (2,3), (1,5), (2,4)]
DEGREE_SCALE = {0: 1., 1: 1e6, 2: 1e9, 3: 1e12, 4: 1e19, 5: 1e23, 6: 1e27, 7: 1e31}

def rotate_local(rx, ry):
    return -ROTATION_SIGN * np.asarray(ry), ROTATION_SIGN * np.asarray(rx)

def model_table():
    rows = []
    original_set, rotated_set = set(X_POWERS), {(b,a) for a,b in Y_POWERS}
    for case, names, powers in [(NATIVE_CASES[0], CPE38_KEYS_DX, X_POWERS), (NATIVE_CASES[1], CPE38_KEYS_DY, Y_POWERS)]:
        for name, (a,b) in zip(names, powers):
            rotated = case == NATIVE_CASES[1]
            px, py = (b,a) if rotated else (a,b)
            sign = ROTATION_SIGN * (-ROTATION_SIGN)**a * ROTATION_SIGN**b if rotated else (-1 if name == "RK5" else 1)
            rows.append(dict(
                case=case, term=name, power_x=px, power_y=py, sign_to_original_x=sign, scale=DEGREE_SCALE[a+b], shared_monomial=(px,py) in original_set & rotated_set,
                ridge_valve=get_ridge_valves([name])[0], span_limit_min=NATIVE_SPAN_LIMITS.get(name, {}).get("min", np.nan), span_limit_max=NATIVE_SPAN_LIMITS.get(name, {}).get("max", np.nan),
            ))
    return pd.DataFrame(rows)

def design_information(matrix):
    norms = np.linalg.norm(matrix, axis=0)
    scaled = matrix / np.where(norms > 0, norms, 1.)
    singular = np.linalg.svd(scaled, compute_uv=False)
    rank = int(np.linalg.matrix_rank(scaled))
    condition = float(singular[0] / singular[-1]) if rank == matrix.shape[1] else np.inf
    return rank, condition

def solve_case(matrix, target, evaluation, names, n_virtual, diagnostic=False):
    if diagnostic:
        norms = np.linalg.norm(matrix, axis=0)
        norms = np.where(norms > 0, norms, 1.)
        beta = np.linalg.lstsq(matrix / norms, target, rcond=None)[0] / norms
        return RidgeFitResult(beta, matrix @ beta, np.zeros(len(names)), np.nan, calc_contribution_spans(evaluation, beta, names), [], 0, 0, "ols_diagnostic_column_scaled")
    return solve_vba_style_ridge_one_axis(
        matrix, target, evaluation, names, NATIVE_SPAN_LIMITS, ridge_regression=RIDGE_REGRESSION, gopt_spec=G_OPT_SPEC, gopt_tolerance=G_OPT_TOLERANCE,
        minimum_ridge=MINIMUM_RIDGE, add_ridge=ADD_RIDGE, max_gopt_return=MAX_GOPT_RETURN, gopt_type=G_OPTIMAL_TYPE,
        inner_or_partial=determine_inner_or_partial(len(target), n_virtual), use_span_box_constraint=USE_SPAN_BOX_CONSTRAINT,
        span_tolerance=SPAN_TOLERANCE, cd_tol=CD_TOL, max_span_iter=MAX_SPAN_ITER, base_lambda_scale=choose_base_lambda_scale(len(target)),
    )

term_mapping = model_table()
save_table("07_TERM_MAPPING", term_mapping)
display(term_mapping)
print("공통 단항식:", int(term_mapping.loc[term_mapping["case"].eq(NATIVE_CASES[0]), "shared_monomial"].sum()), "/ X21과 회전 Y17은 포함관계가 아닙니다.")
print("설정:", "저장 설정 반영" if saved_settings else "원본 노트북 기본값", "; 회전:", ROTATION_DEG)


## 08. 공통 평가 영역 / 물리적 X 기준 행렬 / SVD helper


In [ ]:
def physical_design(x, y, case):
    if case == NATIVE_CASES[0]:
        return cpe_38para(x,y)[0]
    u, v = rotate_local(x,y)
    return ROTATION_SIGN * cpe_38para(u,v)[1]

def scaled_ols_operator(matrix):
    norms = np.linalg.norm(matrix, axis=0)
    norms = np.where(norms > 0, norms, 1.)
    scaled = matrix / norms
    u, singular, vt = np.linalg.svd(scaled, full_matrices=False)
    keep = singular > (singular[0] * SVD_RCOND if len(singular) else 0)
    inverse = np.zeros_like(singular)
    inverse[keep] = 1/singular[keep]
    operator = ((vt.T * inverse) @ u.T) / norms[:,None]
    rank = int(keep.sum())
    return operator, scaled, singular, rank

def hull_metrics(coordinates, grid):
    unique = np.unique(coordinates, axis=0)
    try:
        hull = ConvexHull(unique)
        inside = Delaunay(unique).find_simplex(grid) >= 0
        area = float(hull.volume)
    except (QhullError, ValueError):
        area, inside = 0., np.zeros(len(grid), dtype=bool)
    return area, inside

def native_fixed_lambda(matrix, values_nm, evaluation, names, lambdas):
    beta, _ = ridge_predict(matrix, values_nm/UM_TO_NM, lambdas)
    spans = calc_contribution_spans(evaluation, beta, names)
    violations = find_span_violations(spans, NATIVE_SPAN_LIMITS, tolerance=SPAN_TOLERANCE)
    iterations = 0
    if RIDGE_REGRESSION and USE_SPAN_BOX_CONSTRAINT and violations:
        lower, upper = build_beta_bounds_from_span(evaluation, names, NATIVE_SPAN_LIMITS)
        beta, _, iterations, _ = projected_coordinate_descent_ridge(matrix, values_nm/UM_TO_NM, lambdas, beta_init=beta, lower=lower, upper=upper, tol=CD_TOL, max_iter=MAX_SPAN_ITER)
        spans = calc_contribution_spans(evaluation, beta, names)
        violations = find_span_violations(spans, NATIVE_SPAN_LIMITS, tolerance=SPAN_TOLERANCE)
    return beta*UM_TO_NM, len(violations), iterations

pitch_rows = []
for mode, work in primary_frames.items():
    for axis, coordinate in [("X", "coordinate_X"), ("Y", "coordinate_Y")]:
        col = f"STEP_PITCH_{axis}"
        values = work[col].dropna().unique() if col in work else np.array([])
        values = values[np.isfinite(values) & (values > 0)]
        if len(values) > 1 and not np.allclose(values, values[0]):
            raise ValueError(f"{mode}: 대표 shot의 {col} 값이 일관되지 않습니다.")
        pitch_rows.append(dict(mode=mode, axis=axis, pitch_um=float(values[0]) if len(values) else np.nan, observed_half_extent_um=float(work[coordinate].abs().max())))
pitch_table = pd.DataFrame(pitch_rows)
if COMMON_SHOT_SIZE_UM is not None:
    half_x, half_y = np.asarray(COMMON_SHOT_SIZE_UM, dtype=float)/2
else:
    half_x, half_y = [max(float(pitch_table.loc[pitch_table.axis.eq(axis),"observed_half_extent_um"].max()), float(pitch_table.loc[pitch_table.axis.eq(axis),"pitch_um"].fillna(0).max())/2) for axis in ["X", "Y"]]
if not np.isfinite([half_x,half_y]).all() or min(half_x,half_y) <= 0:
    raise ValueError("공통 평가 영역을 만들 수 없습니다. COMMON_SHOT_SIZE_UM을 지정하세요.")
xx, yy = np.meshgrid(np.linspace(-half_x,half_x,COMMON_GRID_N), np.linspace(-half_y,half_y,COMMON_GRID_N))
grid_coordinates = np.column_stack([xx.ravel(), yy.ravel()])
grid_table = pd.DataFrame(grid_coordinates, columns=COORDS)
inside_hulls = {}
for mode, work in primary_frames.items():
    _, inside_hulls[mode] = hull_metrics(work[COORDS].to_numpy(float), grid_coordinates)
    grid_table[f"inside_{mode}_hull"] = inside_hulls[mode]
shared_hull_mask = inside_hulls["ADI"] & inside_hulls["OCO"]
grid_table["inside_shared_hull"] = shared_hull_mask
save_table("08_COMMON_GRID", grid_table)
save_table("08_PITCH_AND_EXTENT", pitch_table)
print("공통 평가 크기 [um]:", (2*half_x,2*half_y), "; 공통 convex hull 내부 grid:", int(shared_hull_mask.sum()), "/", len(grid_table))


## 09. Sampling별 항 식별성 / alias / noise 전달 계산


In [ ]:
geometry_rows, term_rows, alias_rows, model_cases = [], [], [], {}
for mode, work in primary_frames.items():
    coordinates = work[COORDS].to_numpy(float)
    rx, ry = coordinates.T
    fcp_x, fcp_y = (float(work[col].iloc[0]) for col in ["fcp_x", "fcp_y"])
    if USE_VIRTUAL_CHIP_POINTS:
        gx, gy, virtual_source, n_virtual_full, n_virtual = get_virtual_points_for_group(work, rx, ry, fcp_x, fcp_y, wafer_radius=VIRTUAL_WAFER_RADIUS)
    else:
        gx, gy = build_eval_grid_from_data(rx, ry, grid_n=CPE_EVAL_GRID_N)
        virtual_source, n_virtual_full, n_virtual = "measured_range_grid", len(gx), len(gx)
    area, _ = hull_metrics(coordinates, grid_coordinates)
    for case in NATIVE_CASES:
        names = CPE38_KEYS_DX if case == NATIVE_CASES[0] else CPE38_KEYS_DY
        matrix = physical_design(rx, ry, case)
        evaluation = physical_design(gx, gy, case)
        common_eval = physical_design(grid_coordinates[:,0], grid_coordinates[:,1], case)
        ols, scaled, singular, rank = scaled_ols_operator(matrix)
        tuned = solve_case(matrix, np.zeros(len(matrix)), evaluation, names, n_virtual)
        normal = matrix.T @ matrix + np.diag(tuned.lambdas)
        try:
            ridge = np.linalg.solve(normal, matrix.T)
        except np.linalg.LinAlgError:
            ridge = np.linalg.pinv(normal) @ matrix.T
        operators = {"OLS_SVD": ols, "GOPT_RIDGE": ridge}
        noise_gains, train_noise_gains = {}, {}
        for method, operator in operators.items():
            hat_grid = common_eval @ operator
            noise_gains[method] = np.sqrt(np.sum(hat_grid**2, axis=1))
            residual_operator = np.eye(len(matrix)) - matrix @ operator
            centered_trace = float(np.sum(residual_operator**2) - np.sum(residual_operator.sum(axis=0)**2)/len(matrix))
            train_noise_gains[method] = np.sqrt(max(centered_trace,0.)/(len(matrix)-1)) if len(matrix)>1 else np.nan
        cosine = scaled.T @ scaled
        metadata = term_mapping.loc[term_mapping["case"].eq(case)].reset_index(drop=True)
        for j, name in enumerate(names):
            other = np.delete(scaled, j, axis=1)
            other_operator = scaled_ols_operator(other)[0]
            residual = scaled[:,j] - other @ other_operator @ scaled[:,j]
            denominator = np.linalg.norm(scaled[:,j])
            fraction = float(np.linalg.norm(residual)/denominator) if denominator > 0 else 0.
            row = metadata.iloc[j]
            term_rows.append(dict(mode=mode, case=case, term=name, power_x=int(row.power_x), power_y=int(row.power_y), monomial=f"x^{int(row.power_x)} y^{int(row.power_y)}", independent_fraction=min(fraction,1.), separable_from_other_terms=fraction > 1e-6, observed_column_norm=float(np.linalg.norm(matrix[:,j]))))
            for k in range(j+1,len(names)):
                if abs(cosine[j,k]) >= ALIAS_COSINE_THRESHOLD:
                    alias_rows.append(dict(mode=mode, case=case, term_a=name, term_b=names[k], cosine=float(cosine[j,k]), abs_cosine=float(abs(cosine[j,k]))))
        geometry_rows.append(dict(
            mode=mode, case=case, n_points=len(work), n_unique_points=len(np.unique(coordinates,axis=0)), n_unique_x=work[COORDS[0]].nunique(), n_unique_y=work[COORDS[1]].nunique(),
            span_x_um=float(np.ptp(rx)), span_y_um=float(np.ptp(ry)), convex_hull_area_um2=area, hull_area_fraction=area/(4*half_x*half_y),
            n_parameters=len(names), rank=rank, coefficient_nullity=len(names)-rank, full_column_rank=rank == len(names),
            scaled_condition=float(singular[0]/singular[-1]) if rank == len(names) else np.inf,
            identifiable_subspace_condition=float(singular[0]/singular[rank-1]) if rank else np.inf, normal_equation_condition=float(np.linalg.cond(normal)),
            n_virtual=n_virtual, n_virtual_full=n_virtual_full, virtual_source=virtual_source, gopt=float(tuned.gopt), lambda_max=float(np.max(tuned.lambdas)),
            OLS_noise_gain_rms=float(np.sqrt(np.mean(noise_gains["OLS_SVD"]**2))), ridge_noise_gain_rms=float(np.sqrt(np.mean(noise_gains["GOPT_RIDGE"]**2))),
            ridge_noise_gain_max=float(np.max(noise_gains["GOPT_RIDGE"])), ridge_train_noise_std_gain=float(train_noise_gains["GOPT_RIDGE"]), source_x_values_available=TARGET in work,
        ))
        model_cases[(mode,case)] = dict(matrix=matrix, evaluation=evaluation, common_eval=common_eval, names=names, operators=operators, lambdas=tuned.lambdas, noise_gains=noise_gains, train_noise_gains=train_noise_gains, cosine=cosine, rank=rank, gopt=tuned.gopt)
sampling_summary, term_identifiability = pd.DataFrame(geometry_rows), pd.DataFrame(term_rows)
alias_pairs = pd.DataFrame(alias_rows, columns=["mode", "case", "term_a", "term_b", "cosine", "abs_cosine"])
for name, table in [("09_SAMPLING_SUMMARY", sampling_summary), ("09_TERM_IDENTIFIABILITY", term_identifiability), ("09_ALIAS_PAIRS", alias_pairs)]:
    save_table(name, table)
display(sampling_summary[["mode","case","n_points","rank","n_parameters","coefficient_nullity","lambda_max","ridge_noise_gain_rms"]])


## 10. 동일한 정답 패턴을 ADI/OCO 좌표에 넣어 복원 성능 비교


In [ ]:
def raw_pattern(x, y, components):
    result = np.zeros(len(x), dtype=float)
    for px, py, weight in components:
        result += weight * (np.asarray(x)/half_x)**px * (np.asarray(y)/half_y)**py
    return result

def error_metrics(truth, prediction, prefix):
    error = np.asarray(truth)-np.asarray(prediction)
    return {f"{prefix}_rmse_nm": float(np.sqrt(np.mean(error**2))), f"{prefix}_mean_nm": float(np.mean(error)), f"{prefix}_3sigma_nm": float(3*np.std(error,ddof=1)) if len(error)>1 else np.nan}

x_power_set = set(X_POWERS)
rotated_y_power_set = {(b,a) for a,b in Y_POWERS}
all_powers = [(px,degree-px) for degree in range(8) for px in range(degree,-1,-1)]
patterns = []
for px,py in all_powers:
    member_x, member_y = (px,py) in x_power_set, (px,py) in rotated_y_power_set
    group = "SHARED" if member_x and member_y else "X21_ONLY" if member_x else "ROT_Y17_ONLY" if member_y else "NEITHER"
    patterns.append(dict(pattern=f"x{px}_y{py}", group=group, components=[(px,py,1.)]))
for weight in np.linspace(0,1,11):
    patterns.append(dict(pattern=f"MIX_y3_x4_{weight:.1f}", group="MIXED", components=[(0,3,float(weight)), (4,0,float(1-weight))]))
rng = np.random.default_rng(RANDOM_SEED)
union_powers = sorted(x_power_set | rotated_y_power_set)
for index in range(RANDOM_MIXTURES):
    components = [(px,py,float(rng.normal()/(1+px+py))) for px,py in union_powers]
    patterns.append(dict(pattern=f"RANDOM_MIX_{index:02d}", group="MIXED", components=components))
pattern_rows, response_rows = [], []
for pattern_index, pattern in enumerate(patterns):
    components = pattern["components"]
    raw_grid = raw_pattern(grid_coordinates[:,0],grid_coordinates[:,1],components)
    norm = float(np.sqrt(np.mean(raw_grid**2)))
    if norm <= 0:
        raise ValueError(f"0 패턴: {pattern['pattern']}")
    scale = SYNTHETIC_SIGNAL_RMS_NM/norm
    truth_grid = raw_grid*scale
    pattern_rows.append(dict(pattern=pattern["pattern"], group=pattern["group"], components=json.dumps(components), signal_rms_nm=SYNTHETIC_SIGNAL_RMS_NM))
    for (mode,case), spec in model_cases.items():
        work = primary_frames[mode]
        truth_measured = raw_pattern(work[COORDS[0]].to_numpy(float),work[COORDS[1]].to_numpy(float),components)*scale
        for method in ["OLS_SVD", "GOPT_RIDGE", "NATIVE_SPAN"]:
            if method == "NATIVE_SPAN":
                beta, violations, iterations = native_fixed_lambda(spec["matrix"],truth_measured,spec["evaluation"],spec["names"],spec["lambdas"])
                noise_rms, train_noise_3sigma = np.nan, np.nan
            else:
                beta = spec["operators"][method] @ truth_measured
                violations, iterations = np.nan, 0
                noise_rms = ASSUMED_POINT_NOISE_NM * float(np.sqrt(np.mean(spec["noise_gains"][method]**2)))
                train_noise_3sigma = 3 * ASSUMED_POINT_NOISE_NM * spec["train_noise_gains"][method]
            predicted_measured, predicted_grid = spec["matrix"] @ beta, spec["common_eval"] @ beta
            row = dict(pattern=pattern["pattern"], group=pattern["group"], mode=mode, case=case, method=method, n_points=len(work), rank=spec["rank"], n_parameters=len(spec["names"]), span_violations_after=violations, span_iterations=iterations)
            row.update(error_metrics(truth_measured,predicted_measured,"train"))
            row.update(error_metrics(truth_grid,predicted_grid,"grid"))
            row["shared_hull_rmse_nm"] = float(np.sqrt(np.mean((truth_grid[shared_hull_mask]-predicted_grid[shared_hull_mask])**2))) if shared_hull_mask.any() else np.nan
            row["noise_prediction_rms_nm"] = noise_rms
            row["expected_grid_rmse_nm"] = float(np.sqrt(row["grid_rmse_nm"]**2+noise_rms**2)) if np.isfinite(noise_rms) else np.nan
            row["train_noise_3sigma_nm"] = train_noise_3sigma
            row["expected_train_3sigma_nm"] = float(np.sqrt(row["train_3sigma_nm"]**2+train_noise_3sigma**2)) if np.isfinite(train_noise_3sigma) else np.nan
            response_rows.append(row)
    if (pattern_index+1) % 20 == 0:
        print(f"동일 패턴 재현: {pattern_index+1}/{len(patterns)}")
pattern_catalogue, pattern_response = pd.DataFrame(pattern_rows), pd.DataFrame(response_rows)
save_table("10_PATTERN_CATALOGUE", pattern_catalogue)
save_table("10_PATTERN_RESPONSE", pattern_response)


## 11. 우열 역전 패턴 / 대표 shot 실측 확인 / 전용 항의 추가 설명력


In [ ]:
interaction_rows, observed_rows, group_gain_rows = [], [], []
for (pattern,method), group in pattern_response.groupby(["pattern","method"], sort=False):
    row = dict(pattern=pattern, group=group["group"].iloc[0], method=method)
    indexed = group.set_index(["mode","case"])
    for mode in MODES:
        for metric in ["grid_rmse_nm","expected_grid_rmse_nm","shared_hull_rmse_nm","train_3sigma_nm","expected_train_3sigma_nm"]:
            row[f"{mode}_delta_{metric}_ROT_minus_X21"] = float(indexed.loc[(mode,NATIVE_CASES[1]),metric]-indexed.loc[(mode,NATIVE_CASES[0]),metric])
    row["matches_observed_noiseless"] = row["ADI_delta_grid_rmse_nm_ROT_minus_X21"] < -1e-6 and row["OCO_delta_grid_rmse_nm_ROT_minus_X21"] > 1e-6
    row["matches_observed_training_3sigma"] = row["ADI_delta_train_3sigma_nm_ROT_minus_X21"] < -1e-6 and row["OCO_delta_train_3sigma_nm_ROT_minus_X21"] > 1e-6
    row["matches_observed_expected_training_3sigma"] = row["ADI_delta_expected_train_3sigma_nm_ROT_minus_X21"] < -1e-6 and row["OCO_delta_expected_train_3sigma_nm_ROT_minus_X21"] > 1e-6
    noisy_values = [row[f"{mode}_delta_expected_grid_rmse_nm_ROT_minus_X21"] for mode in MODES]
    row["noise_metric_available"] = bool(np.isfinite(noisy_values).all())
    row["matches_observed_with_assumed_noise"] = noisy_values[0] < -1e-6 and noisy_values[1] > 1e-6 if row["noise_metric_available"] else False
    interaction_rows.append(row)
sampling_interaction = pd.DataFrame(interaction_rows)
if RUN_OBSERVED_CHECK:
    for mode, work in primary_frames.items():
        if TARGET not in work or not np.isfinite(work[TARGET]).all():
            print(mode, ": CPE 직전 실측 X가 없어 좌표 진단만 수행합니다.")
            continue
        observed = work[TARGET].to_numpy(float)*UM_TO_NM
        common_mask = term_mapping.loc[term_mapping["case"].eq(NATIVE_CASES[0]),"shared_monomial"].to_numpy(bool)
        common_matrix = model_cases[(mode,NATIVE_CASES[0])]["matrix"][:,common_mask]
        common_operator, _, _, common_rank = scaled_ols_operator(common_matrix)
        common_residual = observed-common_matrix @ common_operator @ observed
        for case in NATIVE_CASES:
            spec = model_cases[(mode,case)]
            ols_residual = observed-spec["matrix"] @ spec["operators"]["OLS_SVD"] @ observed
            group_gain_rows.append(dict(mode=mode, case=case, n_points=len(work), common_rank=common_rank, full_rank=spec["rank"], added_fit_dof=spec["rank"]-common_rank, common_sse_nm2=float(common_residual @ common_residual), full_sse_nm2=float(ols_residual @ ols_residual), incremental_sse_reduction_nm2=float(common_residual @ common_residual-ols_residual @ ols_residual)))
            for method in ["OLS_SVD", "GOPT_RIDGE", "NATIVE_SPAN"]:
                if method == "NATIVE_SPAN":
                    beta, violations, iterations = native_fixed_lambda(spec["matrix"],observed,spec["evaluation"],spec["names"],spec["lambdas"])
                else:
                    beta = spec["operators"][method] @ observed
                    violations, iterations = np.nan, 0
                row = dict(mode=mode, case=case, method=method, n_points=len(work), span_violations_after=violations, span_iterations=iterations)
                row.update(error_metrics(observed,spec["matrix"] @ beta,"residual"))
                observed_rows.append(row)
observed_check = pd.DataFrame(observed_rows, columns=["mode","case","method","n_points","span_violations_after","span_iterations","residual_rmse_nm","residual_mean_nm","residual_3sigma_nm"])
observed_group_gain = pd.DataFrame(group_gain_rows, columns=["mode","case","n_points","common_rank","full_rank","added_fit_dof","common_sse_nm2","full_sse_nm2","incremental_sse_reduction_nm2"])
for name, table in [("11_SAMPLING_INTERACTION",sampling_interaction),("11_OBSERVED_CHECK",observed_check),("11_OBSERVED_GROUP_GAIN",observed_group_gain)]:
    save_table(name,table)
matching_examples = sampling_interaction.loc[sampling_interaction.matches_observed_noiseless | sampling_interaction.matches_observed_with_assumed_noise | sampling_interaction.matches_observed_training_3sigma | sampling_interaction.matches_observed_expected_training_3sigma]
display(matching_examples.head(PREVIEW_ROWS))
print("같은 정답 패턴으로 ADI는 Y17 / OCO는 X21 우열을 재현한 예시:", len(matching_examples), "행")
print("예시 개수는 원인 확률이 아닙니다. 실측 원인은 좌표·회귀 조건과 실제 오차 패턴을 함께 확인해야 합니다.")


## 12. Sampling 지도 / alias / noise 증폭 / 항별 복원 오차 그림


In [ ]:
plot_paths = []
def finish_figure(fig,name):
    fig.tight_layout()
    path = PLOT_DIR / f"{name}.png"
    fig.savefig(path,dpi=160,bbox_inches="tight")
    fig.savefig(path.with_suffix(".pdf"),bbox_inches="tight")
    plot_paths.append(str(path))
    if SHOW_PLOTS:
        display(fig)
    plt.close(fig)

fig, axes = plt.subplots(2,2,figsize=(12,10))
wafer_extent = 1.06*max(VIRTUAL_WAFER_RADIUS/1000,float(shot_layouts[["fcp_x","fcp_y"]].abs().max().max())/1000)
for column, mode in enumerate(MODES):
    shots = shot_layouts.loc[shot_layouts["mode"].eq(mode)]
    ax = axes[0,column]
    ax.scatter(shots.fcp_x/1000,shots.fcp_y/1000,s=16,color="tab:blue",label="Shot centers")
    work = primary_frames[mode]
    ax.scatter(work.fcp_x.iloc[0]/1000,work.fcp_y.iloc[0]/1000,s=130,marker="*",color="tab:orange",label="Representative shot")
    ax.add_patch(plt.Circle((0,0),VIRTUAL_WAFER_RADIUS/1000,fill=False,color="gray",linestyle=":",lw=.8,label="Virtual-point radius"))
    ax.set(xlabel="Wafer X [mm]",ylabel="Wafer Y [mm]",title=f"{mode}: selected wafer",xlim=(-wafer_extent,wafer_extent),ylim=(-wafer_extent,wafer_extent))
    ax.set_aspect("equal",adjustable="box")
    ax.legend(fontsize=8)
    ax = axes[1,column]
    counts = work.groupby(COORDS).size().reset_index(name="count")
    ax.scatter(counts[COORDS[0]]/1000,counts[COORDS[1]]/1000,s=20+counts["count"]*10,color="tab:blue",label="Measured coordinates")
    ax.set(xlim=(-half_x/1000,half_x/1000),ylim=(-half_y/1000,half_y/1000),xlabel="Local X [mm]",ylabel="Local Y [mm]",title=f"{mode}: {len(work)} rows / {len(counts)} unique points")
    ax.set_aspect("equal",adjustable="box")
    ax.legend(fontsize=8)
for ax in axes.ravel():
    ax.grid(alpha=.2)
finish_figure(fig,"01_sampling_layout")

fig, axes = plt.subplots(2,2,figsize=(13,11))
for row,mode in enumerate(MODES):
    for column,case in enumerate(NATIVE_CASES):
        spec,ax = model_cases[(mode,case)],axes[row,column]
        im = ax.imshow(np.abs(spec["cosine"]),vmin=0,vmax=1,cmap="viridis")
        ax.set_xticks(range(len(spec["names"])),spec["names"],rotation=90,fontsize=7)
        ax.set_yticks(range(len(spec["names"])),spec["names"],fontsize=7)
        ax.set_title(f"{mode} / {case}: rank {spec['rank']}/{len(spec['names'])}")
        fig.colorbar(im,ax=ax,label="Absolute column cosine")
finish_figure(fig,"02_term_alias")

fig, axes = plt.subplots(2,2,figsize=(12,10))
vmax = max(float(spec["noise_gains"]["GOPT_RIDGE"].max()) for spec in model_cases.values())
for row,mode in enumerate(MODES):
    for column,case in enumerate(NATIVE_CASES):
        spec,ax = model_cases[(mode,case)],axes[row,column]
        im = ax.imshow(spec["noise_gains"]["GOPT_RIDGE"].reshape(xx.shape),origin="lower",extent=(-half_x/1000,half_x/1000,-half_y/1000,half_y/1000),vmin=0,vmax=max(vmax,1e-12),cmap="magma",aspect="equal")
        work = primary_frames[mode]
        ax.scatter(work[COORDS[0]]/1000,work[COORDS[1]]/1000,s=9,facecolors="none",edgecolors="cyan",label="Sampling")
        ax.set(xlabel="Local X [mm]",ylabel="Local Y [mm]",title=f"{mode} / {case}\nG-opt ridge before span/box")
        ax.legend(fontsize=8)
        fig.colorbar(im,ax=ax,label="Prediction noise / point noise")
finish_figure(fig,"03_noise_amplification")

single_patterns = pattern_catalogue.loc[~pattern_catalogue.group.eq("MIXED"),"pattern"].tolist()
fig, axes = plt.subplots(1,2,figsize=(13,12))
for ax,method in zip(axes,["OLS_SVD","GOPT_RIDGE"]):
    table = pattern_response.loc[pattern_response.method.eq(method)].pivot(index="pattern",columns=["mode","case"],values="expected_grid_rmse_nm")
    columns = [(mode,case) for mode in MODES for case in NATIVE_CASES]
    matrix = table.loc[single_patterns,columns].to_numpy(float)
    im = ax.imshow(matrix,aspect="auto",cmap="magma")
    ax.set_yticks(range(len(single_patterns)),single_patterns,fontsize=8)
    ax.set_xticks(range(4),[f"{mode}\n{case}" for mode,case in columns],fontsize=8)
    ax.set_title(f"{method}: sqrt(E[grid MSE]), signal {SYNTHETIC_SIGNAL_RMS_NM:g} nm RMS")
    fig.colorbar(im,ax=ax,label=f"nm (assumed point noise {ASSUMED_POINT_NOISE_NM:g} nm)")
finish_figure(fig,"04_pattern_reconstruction")

colors = {"SHARED":"tab:gray","X21_ONLY":"tab:blue","ROT_Y17_ONLY":"tab:orange","NEITHER":"tab:purple","MIXED":"tab:green"}
for metric,label,filename in [("train_3sigma_nm","sample residual 3sigma","05_sample_residual_interaction"),("grid_rmse_nm","common-grid RMSE","06_grid_reconstruction_interaction")]:
    fig, axes = plt.subplots(1,3,figsize=(16,5))
    fig.suptitle("Identical noiseless truth field at both samplings")
    x_col,y_col = [f"{mode}_delta_{metric}_ROT_minus_X21" for mode in MODES]
    for ax,method in zip(axes,["OLS_SVD","GOPT_RIDGE","NATIVE_SPAN"]):
        table = sampling_interaction.loc[sampling_interaction.method.eq(method)]
        for group,color in colors.items():
            points = table.loc[table.group.eq(group)]
            ax.scatter(points[x_col],points[y_col],label=group,color=color,s=23,alpha=.75)
        examples = table.loc[table[x_col].lt(-1e-6) & table[y_col].gt(1e-6)].copy()
        examples["strength"] = examples[x_col].abs()+examples[y_col].abs()
        for row in examples.nlargest(3,"strength").to_dict("records"):
            ax.annotate(row["pattern"],(row[x_col],row[y_col]),xytext=(4,4),textcoords="offset points",fontsize=7)
        ax.axhline(0,color="gray",lw=.7)
        ax.axvline(0,color="gray",lw=.7)
        ax.set(xlabel=f"ADI: rotated Y17 - X21 {label} [nm]",ylabel=f"OCO: rotated Y17 - X21 {label} [nm]",title=f"{method}\nUpper-left: ADI Y17 / OCO X21 preferred")
        ax.grid(alpha=.15)
        ax.legend(fontsize=6)
    finish_figure(fig,filename)


## 13. 진단 Excel / 재현 설정 / 입력·출력 목록 저장


In [ ]:
notes = pd.DataFrame([
    ("scope","같은 critical X의 X21 / 회전 Y17을 비교. Y 실측값은 사용하지 않음. 각 mode 대표 wafer/대표 shot 하나의 좌표 진단."),
    ("selection","가능하면 같은 wafer 선택. 가장 많은 고유 mark를 갖는 반복 layout에서 wafer 중심에 가까운 shot 선택. 전체 layout 목록 별도 저장."),
    ("geometry_only","좌표는 관측 가능성과 노이즈 전달을 결정하지만 실제 오차 패턴의 원인을 단독으로 확정하지 못함."),
    ("same_truth","같은 정답 field를 두 sampling에서 평가하고 공통 grid에서 복원 오차 비교. ADI/OCO 간 point 1:1 매칭을 하지 않음."),
    ("synthetic_units","합성 정답과 오차는 nm. 각 정답 field는 공통 grid RMS로 정규화. 좌표 µm. native span 계산 시 nm를 µm로 변환."),
    ("grid","공통 영역은 두 sampling의 pitch/좌표 범위를 포함하는 직사각형. hull 밖은 외삽 진단. shared_hull_rmse는 두 sampling hull 교집합만 평가."),
    ("OLS_SVD","열 정규화 SVD 최소노름 회귀. rank 부족이면 측정점 residual이 작아도 전체 field를 유일하게 복원할 수 없음."),
    ("GOPT_RIDGE","원본 G-opt로 정한 lambda의 선형 ridge. span/box 이전 단계. 원본 normal equation 계산 방식 사용."),
    ("NATIVE_SPAN","동일 lambda에서 원본 span 검사와 box CD를 적용. 좌표로 정해지는 lambda를 재사용하며 원본 한 방향 solver와 동일한 순서."),
    ("noise","독립·동일 분산 point noise 가정. 실제 noise 추정값 아님. Noise gain은 선형 OLS/ridge에 대한 값. rank 부족의 미관측 field 불확실성은 포함하지 않음."),
    ("expected_grid_rmse","sqrt(무noise grid MSE + sigma_noise^2 * mean(sum(H_grid^2))). sqrt(E[MSE])이며 box 이후에는 계산하지 않음."),
    ("expected_train_3sigma","3*sqrt(E[sample residual variance]). 같은 측정점에서의 signal 잔차 분산 + 가정한 noise 잔차 분산. grid 복원 오차와 구분. box 이후에는 계산하지 않음."),
    ("independent_fraction","한 basis 열을 다른 모든 열로 회귀한 후 남는 상대 norm. 0에 가까우면 해당 계수의 개별 식별이 어려움. alias는 두 열 cosine이므로 다중 항 의존성은 rank도 확인."),
    ("matches_observed","같은 가상 field의 ADI Y17 우세/OCO X21 우세 여부. training_3sigma는 기존 그림과 같은 측정점 잔차 지표, noiseless/with_assumed_noise는 공통 grid RMSE 기준. 예시 개수는 원인 확률이 아님."),
    ("observed_check","입력에 CPE 직전 실측 X가 있을 때 대표 shot에서만 재회귀. 전체 wafer 결과를 대체하지 않음. 실측 Y 불필요."),
    ("observed_gain","공통 12항 OLS 대비 각 모델 전용 항의 추가 측정점 설명력. rank 증가가 0이면 추가 항이 관측 공간을 늘리지 못할 수 있음. noise도 포함할 수 있음."),
    ("missing_shot_geometry","pitch/chip metadata가 없으면 원본과 같이 측정 범위 virtual grid 사용. sampling별 평가영역 차이는 sampling_summary에 표시."),
],columns=["item","description"])
report_path = OUTPUT_DIR / "CPE_Sampling_Diagnostics.xlsx"
report_tables = {
    "Sampling_summary":sampling_summary,"Selected_wafers":selection,"Layout_catalogue":layout_catalogue,"Term_identifiability":term_identifiability,"Alias_pairs":alias_pairs,
    "Pattern_response":pattern_response,"Sampling_interaction":sampling_interaction,"Observed_check":observed_check,"Observed_group_gain":observed_group_gain,
    "Model_terms":term_mapping,"Pattern_catalogue":pattern_catalogue,"Pitch_and_extent":pitch_table,"Notes":notes,
}
with pd.ExcelWriter(report_path,engine="openpyxl") as writer:
    for sheet,table in report_tables.items():
        excel_safe(table).to_excel(writer,sheet_name=sheet,index=False)
        ws = writer.book[sheet]
        ws.freeze_panes = "C2"
        ws.auto_filter.ref = ws.dimensions
        for column in ws.columns:
            ws.column_dimensions[column[0].column_letter].width = min(45,max(16,len(str(column[0].value))+2))
settings = dict(
    rotation_deg=ROTATION_DEG,wafer_selection=WAFER_SELECTION,filters=FILTERS,svd_rcond=SVD_RCOND,common_grid_n=COMMON_GRID_N,common_shot_size_um=[2*half_x,2*half_y],
    signal_rms_nm=SYNTHETIC_SIGNAL_RMS_NM,assumed_point_noise_nm=ASSUMED_POINT_NOISE_NM,random_seed=RANDOM_SEED,random_mixtures=RANDOM_MIXTURES,
    solver_overrides=SOLVER_OVERRIDES,solver={name:globals()[name] for name in sorted(set(SETTINGS_MAP.values()))},active_span_limits=NATIVE_SPAN_LIMITS,plots=plot_paths,
)
(OUTPUT_DIR / "sampling_settings.json").write_text(json.dumps(settings,ensure_ascii=False,indent=2),encoding="utf-8")
pd.DataFrame(input_manifest).to_csv(OUTPUT_DIR / "input_manifest.csv",index=False,encoding="utf-8-sig")
pd.DataFrame(output_manifest.values()).to_csv(OUTPUT_DIR / "saved_files.csv",index=False,encoding="utf-8-sig")
display(FileLink(os.path.relpath(report_path,Path.cwd())))
print("sampling 진단 완료:",report_path)
